In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, RobustScaler

# Open and read the personalized data file.
try:
    df = pd.read_csv("../data/bank_personalized.csv")
    print("Dataset loaded successfully.")

except FileNotFoundError:
    print("Dataset file not found.")

except pd.errors.EmptyDataError:
    print("Dataset file is empty.")

except Exception as e:
    print(f"Unexpected error: {e}")


Dataset loaded successfully.


In [2]:
# Checking the data, separating the features from the target, and splitting it into training and testing sets.

RANDOM_STATE = 202300404

assert set(df["y"].dropna().unique()) ==  {"no", "yes"}

# Create 'X' which has the features the model will use to make predictions. Excluding 'y' since its the target,
# and 'duration' since it's only known post-call, meanwhile we are predicting whether a customer will subscribe pre-call.
X = df.drop(columns=["y", "duration"])

# Convert the target class into numerical values for binary classification.
y = df["y"].map({"no": 0, "yes": 1})
# Check that each value in y is either 'yes' or 'no'; not missing, and not some other third value.
assert y.notna().all()

# Create the stratified train/test split.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Assuming the original dataset has  unique index values, make sure no row index appears in both training and testing sets.
assert X_train.index.intersection(X_test.index).empty

# Make sure the train and test number of rows add up to tghe original total number of rows.
assert len(X_train) + len(X_test) == len(X)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print("\nTraining target distribution (%):")
print(y_train.value_counts(normalize=True).mul(100).round(2))
print("\nTesting target distribution (%):")
print(y_test.value_counts(normalize=True).mul(100).round(2))


Training set: (28934, 15)
Testing set: (7234, 15)

Training target distribution (%):
y
0    88.3
1    11.7
Name: proportion, dtype: float64

Testing target distribution (%):
y
0    88.31
1    11.69
Name: proportion, dtype: float64


Feature Engineering Decisions:
1. pdays_log1p, campaign_log1p, previous_log1p = Logarithmic transformations using np.log1p, which calculates log(1+x). These transformations reduce skewness and the influence of unusually large values. For pdays, we first replace -1 (never contacted) with 0 before applying the transformation.

2. previously_contacted = A new binary feature created from pdays, where 0 means never contacted (pdays = -1) and 1 means previously contacted. This helps the model distinguish between customers who have and have not been contacted before, since replacing -1 with 0 removes this distinction from pdays_log1p.



In [3]:
# Create a function for feature engineering so that it's reproducable & can be used for both the train & test splits.

def engineer_features(data):
    data = data.copy()

    # Assertions rules: campaign & previous must have all values of >=0.
    assert data["campaign"].ge(0).all()
    assert data["previous"].ge(0).all()
    # pdays can = -1 OR can be >=0.
    assert (data["pdays"].eq(-1) | data["pdays"].ge(0)).all()


    cleaned_pdays = data["pdays"].replace(-1, 0)                       # Replace -1 with 0 in a separate series before applying log1p.
    data["pdays_log1p"] = np.log1p(cleaned_pdays)                      # Log transform cleaned_pdays.
    data["previously_contacted"] = data["pdays"].ne(-1).astype(int)    # Converts pdays into a boolean 1/0 column.
    data["campaign_log1p"] = np.log1p(data["campaign"])                # Log transform campaign
    data["previous_log1p"] = np.log1p(data["previous"])                # Log transform previous

    # Confirm the dataset is internally consistent: checks whether the customer had more than zero previous contacts 
    # -> converts those results into 1 and 0 -> checks whether both methods produce the same answer.
    assert (data["previously_contacted"] == data["previous"].gt(0).astype(int)).all()

    # Drop the pre-engineering columns.
    return data.drop(columns=["pdays", "campaign", "previous"])


# Apply the function to the train and test feature sets (X and x).
X_train_fe = engineer_features(X_train)
X_test_fe = engineer_features(X_test)

display(X_train_fe[[
    "poutcome",
    "previously_contacted",
    "campaign_log1p",
    "previous_log1p",
    "pdays_log1p",
]].head())


,poutcome,previously_contacted,campaign_log1p,previous_log1p,pdays_log1p
2792,unknown,0,0.693147,0.0,0.0
23817,unknown,0,0.693147,0.0,0.0
19672,unknown,0,1.791759,0.0,0.0
1416,unknown,0,1.386294,0.0,0.0
12303,unknown,0,1.609438,0.0,0.0


In [4]:
# 1. Separate the features into 3 lists & double-check that no fetaures were forgotten.
categorical_features = [
    "job", "marital", "education", "default", "housing", "loan",
    "contact", "month", "poutcome",
]
numerical_features = [
    "age", "balance", "day", "campaign_log1p", "previous_log1p",
    "pdays_log1p",
]
binary_features = ["previously_contacted"]

assert set(categorical_features + numerical_features + binary_features) == set(X_train_fe.columns)


# 2. Categorical features pipeline: 
# SimpleImputer fills missing values using the most common value in that column.
# OneHotEncoder converts categories into many binary columns (since the categories have no ordinal meaning).
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# 3. Numerical features pipeline:
# Fill missing values using median since its more resistant to outliers.
# RobustScaler scales numerical columns using median and IQR, making scaling less sensitive to outliers than StandardScaler.
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler()),
])

# 4. Combine the pipelines.
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, categorical_features),
        ("numerical", numerical_pipeline, numerical_features),
        ("binary", "passthrough", binary_features), # Binary columns remain unchanged.
    ],
    remainder="drop", # Drops any columns not mentioned.
)

# 5. Create the final pipeline. 
final_pipeline = Pipeline([
    # validate = False prevents a numPy array being created because the engineer_features function expects a pandas DF.
    ("feature_engineering_step1", FunctionTransformer(engineer_features, validate=False)),
    ("preprocessing_step2", preprocessor),
])

print("Pipeline created successfully.")

Pipeline created successfully.


In [5]:
# Applying the preprocessing pipeline.

# Run the final pipeline on the training data, learns the preprocessing parameters & apply all transformations.
X_train_processed = final_pipeline.fit_transform(X_train)
# Apply the preprocessing transformations learned from the training data to the testing data, without learning anything new.
X_test_processed = final_pipeline.transform(X_test)
# Retrieve the preprocessing step from the already-fitted pipeline and stores so we can access its learned information.
fitted_preprocessor = final_pipeline.named_steps["preprocessing_step2"]
# Retrieve the names of all features after preprocessing, including the new columns created by one-hot encoding.
feature_names = fitted_preprocessor.get_feature_names_out()


# Double-check finite values in train & test data before modelling.
assert np.isfinite(X_train_processed).all()
assert np.isfinite(X_test_processed).all()


print("Original training shape:", X_train.shape)
print("Engineered training shape:", X_train_fe.shape)
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)
print("Processed features:", len(feature_names))
print("Training missing values:", np.isnan(X_train_processed).sum())
print("Testing missing values:", np.isnan(X_test_processed).sum())


Original training shape: (28934, 15)
Engineered training shape: (28934, 16)
Processed training shape: (28934, 51)
Processed testing shape: (7234, 51)
Processed features: 51
Training missing values: 0
Testing missing values: 0
